# OmniParser v2 — 실습 데모

GUI 스크린샷에서 UI 요소(텍스트·아이콘)를 자동 감지·레이블링하는 도구입니다.

| 환경 | 준비 사항 |
|------|-----------|
| **VSCode** | Python 가상환경 선택 → 셀 순서대로 실행 |
| **Colab** | 런타임 유형 → T4 GPU 선택 → 셀 순서대로 실행 |

## 0. 환경 설정

In [ ]:
import os, sys, subprocess

# ── 실행 환경 감지 ──────────────────────────────────────────
try:
    import google.colab  # noqa
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# ── 레포 경로 설정 ──────────────────────────────────────────
if IS_COLAB:
    REPO_DIR = '/content/OmniParser'
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ['git', 'clone', 'https://github.com/scythe0425/OmniParser', REPO_DIR],
            check=True
        )
        print('✅ 클론 완료')
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'])
        print('✅ 최신 코드 업데이트')
else:
    # VSCode: 노트북이 레포 루트에 있음 → CWD 기준
    REPO_DIR = os.getcwd()

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'환경      : {"Google Colab" if IS_COLAB else "VSCode / Local"}')
print(f'레포 경로 : {REPO_DIR}')

# ── GPU 확인 ────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU       : {props.name}  ({props.total_memory / 1024**3:.1f} GB VRAM)')
    print(f'CUDA      : {torch.version.cuda}')
else:
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if smi.returncode != 0:
        print('⚠️  GPU 없음 — CPU 모드로 실행합니다 (속도 저하 예상)')
    else:
        print('⚠️  nvidia-smi 확인됨 but torch CUDA 미인식 — 드라이버/CUDA 버전 확인 필요')

## 1. 패키지 설치

> 최초 1회만 실행합니다. 재실행 시 이 셀은 건너뛰어도 됩니다.

In [ ]:
import subprocess, sys

packages = [
    'torch', 'torchvision',       # Colab은 이미 설치돼 있어 no-op, 로컬은 신규 설치
    'easyocr',
    'transformers',
    'ultralytics==8.3.70',
    'supervision==0.18.0',
    'timm',
    'einops==0.8.0',
    'accelerate',
    'paddlepaddle',
    'paddleocr',
    'huggingface_hub',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *packages],
    check=True
)
print('✅ 패키지 설치 완료')

## 2. 모델 가중치 다운로드

파일이 이미 존재하면 건너뜁니다. (~1.6 GB 총 다운로드)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs('weights/icon_detect', exist_ok=True)
os.makedirs('weights/icon_caption_florence', exist_ok=True)

# ── OmniParser-v2.0 가중치 ──────────────────────────────────
MODEL_FILES = {
    'icon_detect/train_args.yaml' : 'weights/icon_detect/train_args.yaml',
    'icon_detect/model.pt'        : 'weights/icon_detect/model.pt',
    'icon_detect/model.yaml'      : 'weights/icon_detect/model.yaml',
    'icon_caption/config.json'            : 'weights/icon_caption_florence/config.json',
    'icon_caption/generation_config.json' : 'weights/icon_caption_florence/generation_config.json',
    'icon_caption/model.safetensors'      : 'weights/icon_caption_florence/model.safetensors',
}

for repo_path, local_path in MODEL_FILES.items():
    if os.path.exists(local_path):
        print(f'  skip  {local_path}')
        continue
    hf_hub_download(
        repo_id='microsoft/OmniParser-v2.0',
        filename=repo_path,
        local_dir='weights/_tmp'
    )
    src = f'weights/_tmp/{repo_path}'
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    shutil.move(src, local_path)
    print(f'  ✓     {local_path}')

shutil.rmtree('weights/_tmp', ignore_errors=True)

# ── Florence-2-base processor + 커스텀 아키텍처 코드 ─────────
# trust_remote_code=True 로컬 로딩 시 .py 파일이 반드시 같은 디렉토리에 있어야 함
PROC_FILES = [
    'tokenizer.json',
    'tokenizer_config.json',
    'preprocessor_config.json',
    'processing_florence2.py',
    'modeling_florence2.py',
    'configuration_florence2.py',
]

for fname in PROC_FILES:
    dest = f'weights/icon_caption_florence/{fname}'
    if os.path.exists(dest):
        print(f'  skip  {dest}')
        continue
    try:
        hf_hub_download(
            repo_id='microsoft/Florence-2-base',
            filename=fname,
            local_dir='weights/icon_caption_florence'
        )
        print(f'  ✓     {dest}')
    except Exception as e:
        print(f'  ⚠️  {fname} 다운로드 실패 (건너뜀): {e}')

print('\n최종 가중치 파일:')
for root, _, files in os.walk('weights'):
    if '.cache' in root or '_tmp' in root:
        continue
    for f in sorted(files):
        print(f'  {os.path.join(root, f)}')

## 3. 모델 로드

In [ ]:
import torch
from PIL import Image
from util.utils import (
    get_som_labeled_img, check_ocr_box,
    get_caption_model_processor, get_yolo_model
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {DEVICE}')

som_model = get_yolo_model('weights/icon_detect/model.pt')
som_model.to(DEVICE)
print('✅ YOLO (icon_detect) 로드 완료')

caption_model_processor = get_caption_model_processor(
    model_name='florence2',
    model_name_or_path='weights/icon_caption_florence',
    device=DEVICE
)
print('✅ Florence2 (icon_caption) 로드 완료')

## 4. 이미지 파싱

`image_path` 에 원하는 스크린샷 경로를 지정하세요.

In [ ]:
import base64, io, time
from PIL import Image

# ── 이미지 선택 ─────────────────────────────────────────────
image_path = 'imgs/windows_home.png'
# image_path = 'imgs/google_page.png'
# image_path = 'imgs/word.png'
# image_path = '/absolute/path/to/screenshot.png'

image = Image.open(image_path)
print(f'이미지: {image_path}  크기: {image.size}')

# ── 파싱 설정 ───────────────────────────────────────────────
ratio = max(image.size) / 3200
draw_cfg = {
    'text_scale'    : 0.8 * ratio,
    'text_thickness': max(int(2 * ratio), 1),
    'text_padding'  : max(int(3 * ratio), 1),
    'thickness'     : max(int(3 * ratio), 1),
}

# ── OCR ─────────────────────────────────────────────────────
t0 = time.time()
(text, ocr_bbox), _ = check_ocr_box(
    image_path,
    display_img=False,
    output_bb_format='xyxy',
    goal_filtering=None,
    easyocr_args={'paragraph': False, 'text_threshold': 0.9},
    use_paddleocr=True
)
print(f'OCR     : {time.time()-t0:.1f}s  ({len(ocr_bbox)}개 텍스트)')

# ── 아이콘 감지 + Florence2 캡션 ────────────────────────────
t1 = time.time()
labeled_img_b64, label_coords, parsed_content_list = get_som_labeled_img(
    image_path, som_model,
    BOX_TRESHOLD=0.05,
    output_coord_in_ratio=True,
    ocr_bbox=ocr_bbox,
    draw_bbox_config=draw_cfg,
    caption_model_processor=caption_model_processor,
    ocr_text=text,
    use_local_semantics=True,
    iou_threshold=0.7,
    scale_img=False,
    batch_size=128
)
print(f'파싱    : {time.time()-t1:.1f}s  ({len(parsed_content_list)}개 요소 감지)')

## 5. 결과 확인

In [ ]:
import base64, io
import matplotlib.pyplot as plt
from PIL import Image

result_img = Image.open(io.BytesIO(base64.b64decode(labeled_img_b64)))
plt.figure(figsize=(16, 9))
plt.imshow(result_img)
plt.axis('off')
plt.title(f'OmniParser v2  —  {len(parsed_content_list)}개 요소 감지', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(parsed_content_list)
df.index.name = 'ID'
print(f'text: {(df.type=="text").sum()}개  icon: {(df.type=="icon").sum()}개')
df